# 04_Embed — Embeddings dense (BGE-M3) + sparse (TF-IDF hashé)

Génère, pour chaque chunk de `chunks_all.jsonl`, un vecteur **dense** BGE-M3
(1024 dimensions, cosine, multilingue AR/FR) et un vecteur **sparse**
(TF-IDF hashé sur tout le corpus — voir `etl_lib/sparse_vectorizer.py`,
module identique à celui utilisé par l'application VM2 en requête, condition
nécessaire pour que le score sparse ait un sens).

In [1]:
import sys, os
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.insert(0, str(Path.cwd()))
from etl_lib.io_utils import load_jsonl, save_jsonl
from etl_lib.sparse_vectorizer import SparseVectorizer

CHUNKS_DIR = Path.cwd().parent / "data" / "chunks"
EMBEDDINGS_DIR = Path.cwd().parent / "data" / "embeddings"
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

chunks = load_jsonl(CHUNKS_DIR / "chunks_all.jsonl")
print(f"Chunks a embedder : {len(chunks)}")


Chunks a embedder : 10257


## 1. Vecteurs denses — BAAI/bge-m3

**Note d'exécution** : cette machine est un poste de travail partagé
(utilisé en parallèle pendant cette session), pas une VM dédiée — le
nombre de threads est volontairement limité pour ne pas aggraver la
contention CPU plutôt que de tenter de saturer tous les cœurs.

In [2]:
import time, os
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
import torch
torch.set_num_threads(4)

from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "BAAI/bge-m3"
t0 = time.time()
model = SentenceTransformer(EMBEDDING_MODEL)
print(f"Modele charge en {time.time()-t0:.0f}s : {EMBEDDING_MODEL} (dimension {model.get_sentence_embedding_dimension()})")

texts = [c["text_bilingual"] for c in chunks]

# Batching manuel + progression ecrite en direct dans un fichier log
# (en plus du print, dont la sortie n'est visible qu'a la fin de la
# cellule quand ce notebook est execute via nbclient/nbconvert) --
# permet de suivre le debit reel depuis l'exterieur du kernel.
PROGRESS_LOG = EMBEDDINGS_DIR / "embed_progress.log"
BATCH = 32
dense_chunks = []
t0 = time.time()
with open(PROGRESS_LOG, "w") as logf:
    for i in range(0, len(texts), BATCH):
        batch_vecs = model.encode(texts[i:i + BATCH], batch_size=BATCH, normalize_embeddings=True)
        dense_chunks.append(batch_vecs)
        done = min(i + BATCH, len(texts))
        elapsed = time.time() - t0
        rate = done / elapsed if elapsed > 0 else 0
        eta_min = (len(texts) - done) / rate / 60 if rate > 0 else 0
        line = f"{done}/{len(texts)} ({rate:.2f} chunks/s, ETA {eta_min:.1f} min)"
        logf.write(line + "\n")
        logf.flush()
        if done % (BATCH * 5) == 0 or done == len(texts):
            print(line)

import numpy as np
dense_vectors = np.concatenate(dense_chunks, axis=0)
print(f"Vecteurs denses generes : {len(dense_vectors)} x {dense_vectors.shape[1]}D en {(time.time()-t0)/60:.1f} min")


/home/ubunto/entraide-mvp/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 22461.38it/s]
/tmp/ipykernel_14540/1805997033.py:12: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Modele charge en {time.time()-t0:.0f}s : {EMBEDDING_MODEL} (dimension {model.get_sentence_embedding_dimension()})")


Modele charge en 6s : BAAI/bge-m3 (dimension 1024)
160/10257 (0.60 chunks/s, ETA 280.0 min)
320/10257 (0.82 chunks/s, ETA 202.1 min)
480/10257 (1.08 chunks/s, ETA 150.2 min)
640/10257 (1.30 chunks/s, ETA 123.1 min)
800/10257 (1.48 chunks/s, ETA 106.3 min)
960/10257 (1.63 chunks/s, ETA 95.3 min)
1120/10257 (1.73 chunks/s, ETA 88.2 min)
1280/10257 (1.82 chunks/s, ETA 82.1 min)
1440/10257 (1.91 chunks/s, ETA 76.8 min)
1600/10257 (1.99 chunks/s, ETA 72.7 min)
1760/10257 (2.05 chunks/s, ETA 69.1 min)
1920/10257 (2.09 chunks/s, ETA 66.5 min)
2080/10257 (2.14 chunks/s, ETA 63.6 min)
2240/10257 (2.19 chunks/s, ETA 61.0 min)
2400/10257 (2.24 chunks/s, ETA 58.5 min)
2560/10257 (2.28 chunks/s, ETA 56.3 min)
2720/10257 (2.33 chunks/s, ETA 54.0 min)
2880/10257 (2.36 chunks/s, ETA 52.0 min)
3040/10257 (2.39 chunks/s, ETA 50.3 min)
3200/10257 (2.43 chunks/s, ETA 48.5 min)
3360/10257 (2.46 chunks/s, ETA 46.8 min)
3520/10257 (2.49 chunks/s, ETA 45.1 min)
3680/10257 (2.52 chunks/s, ETA 43.6 min)
3840/10

## 2. Vecteurs sparse — TF-IDF hashé (cohérent avec l'app VM2)

In [3]:
idf = SparseVectorizer.fit_idf(texts)
vectorizer = SparseVectorizer(idf)
vectorizer.save(EMBEDDINGS_DIR / "sparse_idf.json")
print(f"IDF calcule sur {len(texts)} textes -> {len(idf)} termes distincts")

sparse_vectors = [vectorizer.vectorize(t) for t in texts]
non_empty = sum(1 for indices, _ in sparse_vectors if indices)
print(f"Vecteurs sparse non vides : {non_empty}/{len(sparse_vectors)}")


IDF calcule sur 10257 textes -> 7456 termes distincts
Vecteurs sparse non vides : 10257/10257


## 3. Sauvegarde — `embeddings_all.jsonl`

In [4]:
records = []
for chunk, dense_vec, (sp_idx, sp_val) in zip(chunks, dense_vectors, sparse_vectors):
    records.append({
        "id": chunk["chunk_id"],
        "vector": dense_vec.tolist(),
        "sparse_indices": sp_idx,
        "sparse_values": sp_val,
        "payload": {
            "text": chunk["text_bilingual"][:500],
            "entity_type": chunk["entity_type"],
            "entity_id": chunk["entity_id"],
            "type": chunk["type"],
            "chunk_type": chunk["chunk_type"],
            "langue": chunk["langue"],
            "population_cible": chunk.get("population_cible", ""),
            "institutions": chunk.get("institutions", []),
        },
    })

save_jsonl(EMBEDDINGS_DIR / "embeddings_all.jsonl", records)

import json as _json
sizes_mb = (EMBEDDINGS_DIR / "embeddings_all.jsonl").stat().st_size / 1024 / 1024
print(f"embeddings_all.jsonl : {len(records)} vecteurs, {sizes_mb:.1f} Mo")
print(f"Dimension dense : {len(records[0]['vector'])}")
print(f"\n\u2705 04_Embed termine. Pret pour 05_Neo4j_Load / 06_Qdrant_Index.")


embeddings_all.jsonl : 10257 vecteurs, 230.3 Mo
Dimension dense : 1024

✅ 04_Embed termine. Pret pour 05_Neo4j_Load / 06_Qdrant_Index.
